# conv-padding-zero — ex2: asymmetric 2-D zero padding by slice assignment

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-padding-zero`. Running the final beacon cell reports progress against the `CNN: Conv zero padding` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Conv zero padding` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-padding-zero`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-padding-zero"
DD_SUBTOPIC = "CNN: Conv zero padding"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv zero-padding — quick refresher

Zero-pad a `(B, IC, H, W)` input by allocating a zero-buffer sized to the padded extent, then assigning the original into the interior slice:
```
out = x.new_zeros(B, IC, top + H + bottom, left + W + right)
out[..., top:top + H, left:left + W] = x
```
**Asymmetric pads.** Each of the four spatial sides can take a DIFFERENT amount. The single-int `padding=k` of `nn.Conv2d` is the symmetric shorthand for `top=bottom=left=right=k`; the 4-tuple form lets you match arbitrary input extents (e.g. 'SAME' padding for an even-stride conv often needs asymmetric padding).

**Use `new_zeros`** (not `t.zeros`) so dtype/device inherit from `x`.

### Exercise 2 — asymmetric 2-D zero padding by slice assignment

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the allocate-zero-buffer-then-assign-interior padding pattern to a 2-D input with four independent side amounts (top, bottom, left, right) producing a `(B, IC, top+H+bottom, left+W+right)` output.
> Keywords: padding, 2d, asymmetric, slice-assign
> ```

**KCs targeted:** `pad-allocate-zero-buffer`, `pad-slice-assign-interior`

Implement `ex2_pad2d_asymmetric(x, top, bottom, left, right)`.

- `x` has shape `(B, IC, H, W)`.
- `top, bottom, left, right` are non-negative ints (possibly different).
- Return shape `(B, IC, top + H + bottom, left + W + right)`.
  - The first `top` rows and last `bottom` rows are exactly zero.
  - The first `left` columns and last `right` columns are exactly zero.
  - The interior `[..., top:top+H, left:left+W]` equals `x`.

**Required approach** (do not use `F.pad`):
1. `out = x.new_zeros(B, IC, top + H + bottom, left + W + right)`.
2. `out[..., top:top + H, left:left + W] = x`.
3. Return `out`.

Use `new_zeros` so dtype and device inherit from `x`. The interior slice assignment is the single load-bearing step.

In [ ]:
def ex2_pad2d_asymmetric(x: Tensor, top: int, bottom: int,
                         left: int, right: int) -> Tensor:
    """Zero-pad x with four independent side amounts."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn.functional as F

    # Tiny correctness check.
    x = t.tensor([[[[1.0, 2.0], [3.0, 4.0]]]])   # (1, 1, 2, 2)
    out = ex2_pad2d_asymmetric(x, top=1, bottom=2, left=3, right=0)
    expected_shape = (1, 1, 1 + 2 + 2, 3 + 2 + 0)
    assert tuple(out.shape) == expected_shape, (
        f'expected shape {expected_shape}, got {tuple(out.shape)}'
    )
    # Interior must equal x.
    assert t.equal(out[..., 1:1+2, 3:3+2], x), 'interior must equal x'
    # All padded entries must be exactly zero.
    mask = t.ones_like(out, dtype=t.bool)
    mask[..., 1:1+2, 3:3+2] = False
    assert (out[mask] == 0).all(), 'padded region must be exactly zero'

    # Symmetric case must match nn.functional.pad (which uses
    # argument order: last-axis pad first → (left, right, top, bottom)).
    rng = t.Generator().manual_seed(2)
    x2 = t.randn(2, 3, 5, 7, generator=rng)
    got = ex2_pad2d_asymmetric(x2, top=2, bottom=2, left=4, right=4)
    want = F.pad(x2, (4, 4, 2, 2), mode='constant', value=0)
    assert t.equal(got, want), 'symmetric pad disagrees with F.pad'

    # Asymmetric case must also match F.pad's 4-tuple.
    got2 = ex2_pad2d_asymmetric(x2, top=1, bottom=3, left=2, right=5)
    want2 = F.pad(x2, (2, 5, 1, 3), mode='constant', value=0)
    assert t.equal(got2, want2), 'asymmetric pad disagrees with F.pad'

    # All-zero pad amounts → identity (no extra zeros).
    got3 = ex2_pad2d_asymmetric(x2, 0, 0, 0, 0)
    assert tuple(got3.shape) == tuple(x2.shape)
    assert t.equal(got3, x2)

    # dtype/device inheritance — int dtype must round-trip.
    xi = t.arange(12).reshape(1, 1, 3, 4)
    got4 = ex2_pad2d_asymmetric(xi, top=1, bottom=0, left=0, right=2)
    assert got4.dtype == xi.dtype, (
        f'dtype not inherited: got {got4.dtype}, expected {xi.dtype}; '
        f'did you use t.zeros instead of x.new_zeros?'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_pad2d_asymmetric(x: Tensor, top: int, bottom: int,
                         left: int, right: int) -> Tensor:
    B, IC, H, W = x.shape
    out = x.new_zeros(B, IC, top + H + bottom, left + W + right)
    out[..., top:top + H, left:left + W] = x
    return out
```

**Four independent side amounts** is the generic case; PyTorch's `nn.Conv2d(padding=k)` shorthand collapses it to the symmetric `top=bottom=left=right=k`. The full 4-tuple is what you need for 'SAME' padding when the conv stride or kernel size doesn't split evenly.

**Argument-order trap.** `F.pad`'s tuple is in REVERSE axis order: `(left, right, top, bottom)` for a 2-D pad. Our function uses natural order (`top, bottom, left, right`) — exposing the trap by making both orderings appear in the test side-by-side. Remembering 'F.pad walks axes inside-out' (last axis first) is the single fact that prevents most padding bugs.

**`x.new_zeros` over `t.zeros`.** The former inherits dtype and device automatically. The test for integer round-trip catches the difference: `t.zeros(...)` defaults to `float32` and would silently demote your int input.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()